In [2]:
import os
import os.path as osp
import sys
from tqdm import tqdm

import cv2
import numpy as np
import trimesh
import matplotlib.pyplot as plt

In [3]:
import plotly.graph_objects as go

In [4]:
sys.path.append("..")

In [5]:
from utils.grasp_utils import (
    convert_4x4_to_7dpose,
    convert_7dpose_to_4x4,
    convert_aligned_to_gripper_pose,
    convert_gripper_to_aligned_pose,
)

from utils.pc_utils import (
    backproject_camera, 
    compute_xyz, 
    load_depth_img,
    filter_outliers
)

/home/ninad/miniconda3/envs/mfg2/lib/python3.10/site-packages/torch/cuda/__init__.py:118: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /opt/conda/conda-bld/pytorch_1716905971132/work/c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [6]:
k = [554.254691191187, 0.0, 320.5, 0.0, 554.254691191187, 240.5, 0.0, 0.0, 1.0]
intrinsics = np.array(k).reshape(3, 3)
fx = intrinsics[0, 0]
fy = intrinsics[1, 1]
px = intrinsics[0, 2]
py = intrinsics[1, 2]

In [7]:
print(intrinsics)

[[554.25469119   0.         320.5       ]
 [  0.         554.25469119 240.5       ]
 [  0.           0.           1.        ]]


# Data Paths

In [8]:
input_dir = "/home/ninad/Datasets/MMDemo/whiteboard-eraser_interval_0.05/"
hamer_root_dir = osp.join(input_dir, "out", "hamer")
hamer_npz_dir = osp.join(hamer_root_dir, "model")
depth_img_dir = osp.join(input_dir, "depth")

In [10]:
samv2_dir = osp.join(input_dir, "out", "samv2")
# NOTE: Assuming only 1 folder within the samv2 directory
masks_dir = osp.join(samv2_dir, os.listdir(samv2_dir)[0], "obj_masks")
print(masks_dir) 

/home/ninad/Datasets/MMDemo/whiteboard-eraser_interval_0.05/out/samv2/black_eraser/obj_masks


In [11]:
npz_files = [
    f
    for f in os.listdir(hamer_npz_dir)
    if osp.isfile(osp.join(hamer_npz_dir, f)) and f.lower().endswith((".npz"))
]
if not npz_files:
    raise ValueError(f"No npz files found in {hamer_npz_dir}!....")


In [12]:
npz_files = sorted(npz_files)
print(npz_files)

['000049.npz', '000050.npz', '000051.npz', '000052.npz', '000053.npz', '000054.npz', '000055.npz', '000056.npz', '000057.npz', '000058.npz', '000059.npz', '000060.npz', '000061.npz', '000062.npz', '000063.npz', '000064.npz', '000065.npz', '000066.npz', '000067.npz', '000068.npz', '000069.npz', '000070.npz', '000071.npz', '000072.npz', '000073.npz', '000074.npz', '000075.npz', '000076.npz', '000077.npz', '000078.npz', '000079.npz', '000080.npz', '000081.npz', '000082.npz', '000083.npz', '000084.npz', '000085.npz', '000086.npz', '000087.npz', '000088.npz', '000089.npz', '000090.npz', '000091.npz', '000092.npz', '000093.npz', '000094.npz', '000095.npz', '000096.npz', '000097.npz', '000098.npz', '000099.npz', '000100.npz', '000101.npz', '000102.npz', '000103.npz', '000104.npz', '000105.npz', '000106.npz', '000107.npz', '000108.npz', '000109.npz', '000110.npz', '000111.npz', '000112.npz', '000113.npz', '000114.npz', '000115.npz', '000116.npz', '000117.npz', '000118.npz', '000119.npz', '0001

In [13]:
# Keeping the first frame id as 1 instead of 0
first_frame_id = "000001"
depth_img_f = osp.join(depth_img_dir, f"{first_frame_id}.png")
depth_im = load_depth_img(depth_img_f)

mask_f = osp.join(masks_dir, f"{first_frame_id}.png")
mask_im = cv2.imread(mask_f, 0)

# scene_pc = compute_xyz(depth_im, fx, fy, px, py)
obj_pc_first_view = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
print(obj_pc_first_view.shape)

(573, 3)


In [14]:
# Function to get the hand - object distance based on:
# Distance between object center and hand palm origin (from aligned pose)
def get_HOdist(obj_pc, RT_gripper, gripper_name="fetch_gripper"):
    pose_7d = convert_4x4_to_7dpose(RT_gripper)
    pose_7d_alg = convert_gripper_to_aligned_pose(pose_7d, gripper_name)
    palm_pos = pose_7d_alg[:3]
    obj_center = np.mean(obj_pc, axis=0)
    # ho_dist = np.sqrt(np.sum((obj_center - palm_pos)**2))
    return np.linalg.norm(obj_center - palm_pos)


In [15]:
num_frames = int(osp.splitext(npz_files[-1])[0]) + 1
print(num_frames)

248


In [16]:
# Initialize an array for distances (for each frame)
# Setting a default value of 1.2 meters

dists_first_view = [1.2] * num_frames

for idx, npz_f in enumerate(npz_files):
    frame_id = osp.splitext(npz_f)[0]
    frame_id_idx = int(frame_id)
    # NOTE: Code commented out below is not used, but kept for future reference
    # depth_img_f = osp.join(depth_img_dir, f"{frame_id}.png")
    # depth_im = load_depth_img(depth_img_f)
    # mask_f = osp.join(masks_dir, f"{frame_id}.png")
    # mask_im = cv2.imread(mask_f, 0)
    # obj_pc = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
    ### point_cloud = trimesh.points.PointCloud(obj_pc)
    ### point_cloud.export(f"{frame_id}_objpc.ply")   

    npz_fpath = osp.join(hamer_npz_dir, npz_f)
    npz_data = dict(
        np.load(npz_fpath, allow_pickle=True)
    )  # load the npz as dict to be able to update later
    RT_grippers = npz_data["target_transfer_pose"]
    _dists = []
    # print(frame_id)
    for i in range(RT_grippers.shape[0]):
        hodist = get_HOdist(obj_pc_first_view, RT_grippers[i])
        # print(i, hodist)
        _dists.append(hodist)
    dists_first_view[frame_id_idx] = min(_dists)

dists_first_view = np.asarray(dists_first_view)
indices = list(range(len(dists_first_view)))

In [17]:
# plt.plot(indices, dists, marker='o', linestyle='-', color='b', label='Distance')
# # Add labels and title
# plt.xlabel('Index')
# plt.ylabel('Distance')
# plt.title('Distance vs Index')
# plt.legend()

# # Show the plot
# plt.grid()
# plt.show()

In [18]:
# Create a Plotly figure
fig = go.Figure()

# Add the data as a line plot with markers
fig.add_trace(go.Scatter(
    x=indices, 
    y=dists_first_view, 
    mode='lines+markers',
    name='Distance',
    line=dict(color='blue'),
    marker=dict(size=8)
))

fig.show()

# Find Gripper Closing Frames

- Iterate over the distances in forward order
- Find local minima over a window of size `k` (here `k` = 2) 

In [19]:
delta = 0.08
k = 2
frames_close = []
for i in range(num_frames):
    if dists_first_view[i] > delta:
        continue
    if abs(dists_first_view[i] - dists_first_view[i+1]) > 0.02:
        continue 
    if dists_first_view[i] < dists_first_view[i - k] and dists_first_view[i] < dists_first_view[i + k]:
        frames_close.append(i)
        if len(frames_close) >= 2:
            break
print(frames_close)


[80, 81]


In [20]:
# last_frame_id = f"{num_frames-1:06}"
# print(last_frame_id)

# Find Gripper Opening Frames

In [21]:
# Find out the last frame ID
last_frame_id = f"{num_frames-1:06}"
depth_img_f = osp.join(depth_img_dir, f"{last_frame_id}.png")
depth_im = load_depth_img(depth_img_f)

mask_f = osp.join(masks_dir, f"{last_frame_id}.png")
mask_im = cv2.imread(mask_f, 0)

# scene_pc = compute_xyz(depth_im, fx, fy, px, py)
# NOTE: Getting the object point cloud as seen in the last frame
obj_pc_last_view = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
print(obj_pc_last_view.shape)

(562, 3)


In [22]:
# Compute distances as before,
# NOTE: but from the object point cloud seen in last frame
dists_last_view = [1.2] * num_frames

for idx, npz_f in enumerate(npz_files):
    frame_id = osp.splitext(npz_f)[0]
    frame_id_idx = int(frame_id)
    npz_fpath = osp.join(hamer_npz_dir, npz_f)
    npz_data = dict(
        np.load(npz_fpath, allow_pickle=True)
    )  # load the npz as dict to be able to update later
    RT_grippers = npz_data["target_transfer_pose"]
    _dists = []
    # print(frame_id)
    for i in range(RT_grippers.shape[0]):
        hodist = get_HOdist(obj_pc_last_view, RT_grippers[i])
        # print(i, hodist)
        _dists.append(hodist)
    dists_last_view[frame_id_idx] = min(_dists)

dists_last_view = np.asarray(dists_last_view)
indices = list(range(len(dists_last_view)))

In [23]:
# Create a Plotly figure
fig = go.Figure()

# Add the data as a line plot with markers
fig.add_trace(go.Scatter(
    x=indices, 
    y=dists_last_view, 
    mode='lines+markers',
    name='Distance',
    line=dict(color='blue'),
    marker=dict(size=8)
))

fig.show()

In [24]:
delta = 0.08
k = 2
frames_open = []
for i in range(num_frames)[::-1]:
    if dists_last_view[i] > delta:
        continue
    if abs(dists_last_view[i] - dists_last_view[i+1]) > 0.02:
        continue 
    if dists_last_view[i] < dists_last_view[i - k] and dists_last_view[i] < dists_last_view[i + k]:
        frames_open.append(i)
        if len(frames_open) >= 2:
            break
print(frames_open)

[220, 219]
